# Part 0: Exporting geological information from regional maps for the Garonne

## Introduction
This notebook extracts lithological class fractions from the BD LISA regional geological map for the 22 Garonne study catchments. The BD LISA shapefile is clipped to each catchment boundary and the 18 lithological classes are reclassified into three permeability groups (high, medium, low). The main output — `estreams_geology_garonne_regional_attributes.csv` — is then consumed by Part-A, Part-B, Part-C, Part-D, and Part-E for all Garonne geology experiments.

Alongside the other notebooks, it covers part of the analysis performed in: "Assessing the Impact of Geological Map Detail on Process-Based and Data-Driven Hydrological Models" by do Nascimento et al. (2026).

Author: Thiago Nascimento (thiago.nascimento@eawag.ch)

## Requirements

**Python packages:** `geopandas`, `pandas`, `numpy`, `rasterio`, `tqdm`, `os`, `time`

Check the repository for `environment.yml` (conda) or `requirements.txt` (pip).

**Third-party GIS data** — download and place under `data/lithology/` before running:

- `data/lithology/garonne_litho_fr_n2_o1_3035_dissolved.shp` — BD LISA hydrogeological map of France (version 1, level 2, order 1, scale 1:250,000). Download at: https://bdlisa.eaufrance.fr/telechargement (Last access: 23 November 2025)

**From this repository (`../data/`):**

- `data/shapefiles/estreams_boundaries.shp` — EStreams catchment boundaries, used to clip the lithological map to the Garonne study area

**Produced by this notebook:**

- `../data/estreams_geology_garonne_regional_attributes.csv` — regional-scale lithological class fractions (18 classes) for the 22 Garonne study catchments, used as input to Part-A and Part-B

**References:**

- BD LISA database (version 1, level 2, order 1, scale: 1:250,000). Available at: https://bdlisa.eaufrance.fr (Last access: 23 November 2025)

**Licenses**

The BD LISA data are provided under Etalab Open License. The original third-party data are not redistributed in this repository due to licensing and storage constraints.


# Import modules

In [1]:
import geopandas as gpd
import pandas as pd
import tqdm as tqdm
import os
import numpy as np
import rasterio
import time
from rasterio.features import geometry_mask

# Configurations

In [ ]:
# Only editable variables:
# Relative path to your local directory
PATH = ".."
path_estreams = "C:/Users/nascimth/Documents/data/"

* #### The users should NOT change anything in the code below here.


In [3]:
PATH_OUTPUT = "results/staticattributes/"
# Set the directory:
os.chdir(PATH)

# Import data
## Catchment boundaries

In [4]:
catchment_boundaries = gpd.read_file(path_estreams+'/EStreams/shapefiles/estreams_catchments.shp')
catchment_boundaries.set_index("basin_id", inplace = True)
catchment_boundaries.head()

,gauge_id,country,area_offic,area_estre,area_flag,area_rel,start_date,end_date,gauge_flag,upstream,group,geometry
basin_id,,,,,,,,,,,,
AT000001,200014,AT,4647.9,4668.379,0,-0.440608,1996-01-01,2021-12-31,B,16,1,"POLYGON Z ((9.69406 46.54322 0.00000, 9.69570 ..."
AT000002,200048,AT,102.0,102.287,0,-0.281373,1958-10-01,2021-12-31,B,1,1,"POLYGON Z ((10.13650 47.02949 0.00000, 10.1349..."
AT000003,231662,AT,535.2,536.299,0,-0.205344,1985-01-02,2021-12-31,B,2,1,"POLYGON Z ((10.11095 46.89437 0.00000, 10.1122..."
AT000004,200592,AT,66.6,66.286,0,0.471471,1998-01-02,2021-12-31,B,1,1,"POLYGON Z ((10.14189 47.09706 0.00000, 10.1404..."
AT000005,200097,AT,72.2,72.448,0,-0.343490,1990-01-01,2019-12-31,B,3,1,"POLYGON Z ((9.67851 47.06249 0.00000, 9.67888 ..."


## Gauges information

In [6]:
network_estreams = pd.read_csv(path_estreams+'/EStreams/streamflow_gauges/estreams_gauging_stations.csv', encoding='utf-8')
network_estreams.set_index("basin_id", inplace = True)
network_estreams

,gauge_id,gauge_name,gauge_country,gauge_provider,river,lon_snap,lat_snap,lon,lat,elevation,...,num_continuous_days,num_days_gaps,num_days_reliable,num_days_noflag,num_days_suspect,gauge_flag,duplicated_suspect,watershed_group,gauges_upstream,nested_catchments
basin_id,,,,,,,,,,,,,,,,,,,,,
AT000001,200014,Bangs,AT,AT_EHYD,Rhein,9.534835,47.273748,9.534835,47.273748,420,...,9497,0.0,0.0,9497.0,0.0,B,['CH000197'],1,16,"['AT000001', 'CH000010', 'CH000046', 'CH000048..."
AT000002,200048,Schruns (Vonbunweg),AT,AT_EHYD,Litz,9.913677,47.080301,9.913677,47.080301,673,...,23103,0.0,0.0,23103.0,0.0,B,['CH000221'],1,1,['AT000002']
AT000003,231662,Loruens-Aeule,AT,AT_EHYD,Ill,9.847765,47.132821,9.847765,47.132821,579,...,13513,0.0,0.0,13513.0,0.0,B,['CH000215'],1,2,"['AT000002', 'AT000003', 'CH000221']"
AT000004,200592,Kloesterle (OEBB),AT,AT_EHYD,Alfenz,10.061843,47.128994,10.061843,47.128994,1014,...,8765,0.0,0.0,8765.0,0.0,B,['CH000227'],1,1,['AT000004']
AT000005,200097,Buers (Bruecke L82),AT,AT_EHYD,Alvier,9.802668,47.150770,9.802668,47.150770,564,...,10957,0.0,0.0,10957.0,0.0,B,['CH000214'],1,3,"['AT000005', 'CH000214']"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
UAGR0017,6682300,BASHTANOVKA,UA,UA_GRDC,KACHA,33.894739,44.691884,33.900000,44.683333,NaN,...,3652,0.0,0.0,3652.0,0.0,B,NaN,1988,1,['UAGR0017']
UAGR0018,6682500,YALTA,UA,UA_GRDC,DERE-KIOY,34.166667,44.500000,34.166667,44.500000,16,...,3652,0.0,0.0,3652.0,0.0,B,NaN,1989,1,['UAGR0018']
UAGR0019,6683010,PIONERSKOE,UA,UA_GRDC,SALHYR,34.199841,44.887685,34.200000,44.883333,307,...,3652,0.0,0.0,3652.0,0.0,B,NaN,1990,1,['UAGR0019']


In [7]:
nested_catchments = pd.DataFrame(network_estreams.nested_catchments)
nested_catchments

,nested_catchments
basin_id,
AT000001,"['AT000001', 'CH000010', 'CH000046', 'CH000048..."
AT000002,['AT000002']
AT000003,"['AT000002', 'AT000003', 'CH000221']"
AT000004,['AT000004']
AT000005,"['AT000005', 'CH000214']"
...,...
UAGR0017,['UAGR0017']
UAGR0018,['UAGR0018']
UAGR0019,['UAGR0019']


In [8]:
# Convert the string representation of lists to actual lists
nested_catchments['nested_catchments'] = nested_catchments['nested_catchments'].apply(eval)
nested_catchments

,nested_catchments
basin_id,
AT000001,"[AT000001, CH000010, CH000046, CH000048, CH000..."
AT000002,[AT000002]
AT000003,"[AT000002, AT000003, CH000221]"
AT000004,[AT000004]
AT000005,"[AT000005, CH000214]"
...,...
UAGR0017,[UAGR0017]
UAGR0018,[UAGR0018]
UAGR0019,[UAGR0019]


In [9]:
catchment_boundaries

,gauge_id,country,area_offic,area_estre,area_flag,area_rel,start_date,end_date,gauge_flag,upstream,group,geometry
basin_id,,,,,,,,,,,,
AT000001,200014,AT,4647.9,4668.379,0,-0.440608,1996-01-01,2021-12-31,B,16,1,"POLYGON Z ((9.69406 46.54322 0.00000, 9.69570 ..."
AT000002,200048,AT,102.0,102.287,0,-0.281373,1958-10-01,2021-12-31,B,1,1,"POLYGON Z ((10.13650 47.02949 0.00000, 10.1349..."
AT000003,231662,AT,535.2,536.299,0,-0.205344,1985-01-02,2021-12-31,B,2,1,"POLYGON Z ((10.11095 46.89437 0.00000, 10.1122..."
AT000004,200592,AT,66.6,66.286,0,0.471471,1998-01-02,2021-12-31,B,1,1,"POLYGON Z ((10.14189 47.09706 0.00000, 10.1404..."
AT000005,200097,AT,72.2,72.448,0,-0.343490,1990-01-01,2019-12-31,B,3,1,"POLYGON Z ((9.67851 47.06249 0.00000, 9.67888 ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
UAGR0017,6682300,UA,321.0,325.370,0,-1.361371,1978-01-01,1987-12-31,B,1,1988,"POLYGON Z ((33.96791 44.63291 0.00000, 33.9679..."
UAGR0018,6682500,UA,49.7,47.594,0,4.237425,1978-01-01,1987-12-31,B,1,1989,"POLYGON Z ((34.19958 44.58291 0.00000, 34.2029..."
UAGR0019,6683010,UA,261.0,244.731,1,6.233333,1978-01-01,1987-12-31,B,1,1990,"POLYGON Z ((34.19624 44.88375 0.00000, 34.1962..."


In [10]:
# Function to filter data_df using lists from nested_catchments
def filter_data(basin_id):
    # Retrieve the list of nested catchments for the given basin_id
    nested_list = nested_catchments.loc[basin_id, 'nested_catchments']
    
    # Filter data_df using the nested list
    filtered_df = network_estreams.loc[nested_list]
    return filtered_df

In [11]:
# Example usage
basin_id = 'FR001604'
nested_clip = filter_data(basin_id)
nested_clip

,gauge_id,gauge_name,gauge_country,gauge_provider,river,lon_snap,lat_snap,lon,lat,elevation,...,num_continuous_days,num_days_gaps,num_days_reliable,num_days_noflag,num_days_suspect,gauge_flag,duplicated_suspect,watershed_group,gauges_upstream,nested_catchments
basin_id,,,,,,,,,,,,,,,,,,,,,
ES000333,9019,BOSSOST,ES,ES_CEDEX,NaN,0.691225,42.782327,0.691225,42.782327,700,...,3287,8305.0,0.0,11418.0,0.0,E,NaN,541,3,"['ES000333', 'ES000453', 'ES000507']"
ES000453,9143,ARTIES,ES,ES_CEDEX,NaN,0.879104,42.701971,0.879104,42.701971,1180,...,6321,8852.0,0.0,15985.0,0.0,E,NaN,541,1,['ES000453']
ES000507,9200,ARTIES,ES,ES_CEDEX,NaN,0.872524,42.695670,0.873007,42.695793,1156,...,4044,0.0,0.0,4044.0,0.0,B,NaN,541,2,"['ES000453', 'ES000507']"
FR004084,O001531001,Le Maudan ÃƒÂ Fos et ÃƒÂ Melles,FR,FR_EAUFRANCE,Le Maudan à Fos et à Melles,0.750000,42.865833,0.747758,42.867300,555,...,17387,110.0,20459.0,29.0,2320.0,C,NaN,541,1,['FR004084']
FR004085,O004401001,La Pique Ã Cier-de-Luchon,FR,FR_EAUFRANCE,La Pique à Cier-de-Luchon,0.603185,42.853382,0.603185,42.853382,580,...,4383,0.0,4211.0,0.0,172.0,A,NaN,541,2,"['FR004085', 'FR001501']"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FR001600,O243402001,O2434020,FR,FR_EAUFRANCE,La Gesse à Boulogne-sur-Gesse,0.664848,43.272530,0.664848,43.272530,NaN,...,3560,0.0,0.0,0.0,3560.0,C,NaN,541,2,"['FR001600', 'FR001601']"
FR001601,O243402101,O2434010,FR,FR_EAUFRANCE,La Gesse [Prise] à Villeneuve-Lécussan,0.465553,43.155092,0.465553,43.155092,550,...,8401,0.0,8401.0,0.0,0.0,A,NaN,541,1,['FR001601']
FR001602,O246293201,O2462920,FR,FR_EAUFRANCE,La Save [CACG] à Lombez [CACG],0.881518,43.453239,0.881518,43.453239,167,...,14754,0.0,14754.0,0.0,0.0,A,NaN,541,6,"['FR001521', 'FR001597', 'FR001599', 'FR001600..."


In [12]:
catchment_boundaries_clip = catchment_boundaries.loc[nested_clip.index.tolist()]
catchment_boundaries_clip

,gauge_id,country,area_offic,area_estre,area_flag,area_rel,start_date,end_date,gauge_flag,upstream,group,geometry
basin_id,,,,,,,,,,,,
ES000333,9019,ES,440.0,445.690,0,-1.293182,1965-10-01,2019-09-30,E,3,541,"POLYGON Z ((0.68874 42.72125 0.00000, 0.68791 ..."
ES000453,9143,ES,137.0,144.472,0,-5.454015,1950-10-01,2018-09-30,E,1,541,"POLYGON Z ((0.87541 42.74291 0.00000, 0.87624 ..."
ES000507,9200,ES,50.0,145.186,999,-190.372000,1980-10-01,1991-10-27,B,2,541,"POLYGON Z ((0.87541 42.74291 0.00000, 0.87624 ..."
FR004084,O001531001,FR,38.0,31.038,1,18.321053,1961-01-01,2023-09-30,C,1,541,"POLYGON Z ((0.77708 42.89458 0.00000, 0.78291 ..."
FR004085,O004401001,FR,300.0,302.892,0,-0.964000,1920-01-01,1931-12-31,A,2,541,"POLYGON Z ((0.56041 42.86125 0.00000, 0.56041 ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
FR001600,O243402001,FR,NaN,43.207,999,NaN,2014-01-01,2023-09-30,C,2,541,"POLYGON Z ((0.64208 43.28041 0.00000, 0.64541 ..."
FR001601,O243402101,FR,0.0,1.996,999,-inf,1970-10-01,1993-09-30,A,1,541,"POLYGON Z ((0.48374 43.16958 0.00000, 0.48791 ..."
FR001602,O246293201,FR,424.0,423.228,0,0.182075,1965-10-01,2006-02-21,A,6,541,"POLYGON Z ((0.67124 43.22458 0.00000, 0.67124 ..."


In [ ]:
#catchment_boundaries_clip = catchment_boundaries[catchment_boundaries.country == "FR"]
#catchment_boundaries_clip

,gauge_id,country,area_offic,area_estre,area_flag,area_rel,start_date,end_date,gauge_flag,upstream,group,geometry
basin_id,,,,,,,,,,,,
FR000001,A021005050,FR,35878.0,35803.773,0,0.206887,2013-11-26,2023-09-30,B,215,1,"POLYGON Z ((7.60958 47.54958 0.00000, 7.60958 ..."
FR000002,A022020001,FR,15.0,16.011,0,-6.740000,1993-01-26,2023-09-30,C,1,1,"POLYGON Z ((7.52708 47.56541 0.00000, 7.52791 ..."
FR000003,A022065001,FR,8.2,8.174,0,0.317073,1993-01-26,1995-05-31,C,1,1,"POLYGON Z ((7.49791 47.58958 0.00000, 7.50041 ..."
FR000004,A023010001,FR,NaN,38.921,999,NaN,1994-01-20,1996-01-09,E,3,1,"POLYGON Z ((7.53458 47.61125 0.00000, 7.53708 ..."
FR000005,A023020001,FR,NaN,34.648,999,NaN,1993-02-05,1993-09-08,A,2,1,"POLYGON Z ((7.55374 47.59125 0.00000, 7.55541 ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
FR003154,Y980000301,FR,53.0,53.335,0,-0.632075,1971-06-01,1989-03-29,A,3,905,"POLYGON Z ((9.17791 41.66791 0.00000, 9.18124 ..."
FR003155,Y980000302,FR,53.0,53.393,0,-0.741509,2020-07-02,2023-09-30,C,3,905,"POLYGON Z ((9.17791 41.66791 0.00000, 9.18124 ..."
FR003156,Y982000101,FR,29.0,29.680,0,-2.344828,1969-01-01,1980-11-30,C,1,906,"POLYGON Z ((9.22208 41.50958 0.00000, 9.22874 ..."


## Fabrizio's high resolution lithology shapefile

In [13]:
garonne_litho_fr = gpd.read_file(path_estreams+'/gis/garonne_litho_fr_n2_o1_3035_dissolved.shp')
garonne_litho_fr

,basin_id,gauge_id,country,area_offic,area_estre,area_flag,area_rel,start_date,end_date,gauge_flag,...,incluseh,etateh,natureeh,milieueh,themeeh,origineeh,layer_2,path_2,litho_name,geometry
0,FR001604,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,C,...,334,3,3,4,2,1,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,"MULTIPOLYGON Z (((1.88291 43.44458 0.00000, 1...."
1,FR001604,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,C,...,699,2,4,4,4,1,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,"MULTIPOLYGON Z (((1.93208 42.60458 0.00000, 1...."
2,FR001604,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,C,...,699,2,3,3,4,1,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,"MULTIPOLYGON Z (((1.99375 42.65208 0.00000, 1...."
3,FR001604,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,C,...,402,3,3,5,4,4,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,"MULTIPOLYGON Z (((2.07791 42.93958 0.00000, 2...."
4,FR001604,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,C,...,404,2,4,2,4,4,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,"MULTIPOLYGON Z (((0.66502 42.91464 0.00000, 0...."
5,FR001604,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,C,...,946,2,3,1,1,4,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,"MULTIPOLYGON Z (((1.60306 42.86921 0.00000, 1...."
6,FR001604,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,C,...,699,2,4,4,4,1,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,"POLYGON Z ((1.85291 42.58125 0.00000, 1.85291 ..."
7,FR001604,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,C,...,318,X,4,1,2,4,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,"MULTIPOLYGON Z (((0.61874 43.27541 0.00000, 0...."
8,FR001604,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,C,...,344,1,3,4,2,4,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,"MULTIPOLYGON Z (((1.61258 43.02182 0.00000, 1...."
9,FR001604,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,C,...,306,2,3,1,2,4,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,"MULTIPOLYGON Z (((1.29458 43.84875 0.00000, 1...."


In [18]:
# Step 1: Define your keyword mapping
litho_mapping = {
    "Alluvions ": "Alluvium",
    "impermeables": "Impermeable",
    "Calcaires ": "Limestones",
    "Calcaires et grès": "Limestones and sandstones",
    "Calcaires, grès et graviers": "Limestones, sandstones and gravels",
    "Calcaires, grès et marnes": "Limestones, sandstones and marls",
    "Chaînons calcaires": "Limestones",
    "Formations cristallines et métamorphiques": "Schists, gneisses, granites",
    "Formations majoritairement carbonatées du Pays de Sault": "Carbonate rocks",
    "Formations molassiques de l'Eocène du bassin de Carcassonne": "Sedimentary rocks",
    "Granitoïdes": "Granitoids",
    "Inland water": "Inland water",
    "Limestones": "Limestones",
    "Magmatic rocks": "Magmatic rocks",
    "Massif": "Massif",
    "Molasses et argiles": "Molasses and clays",
    "Plutonic rocks": "Plutonic rocks",
    "glaises": "Sands and clays",
    "Shales": "Shales",
    "Sédiments mésozoiques": "Sediments mesosoics",
    "Terrasses": "Sand, gravel and pebbles",
}

# Step 2: Function to assign standardized name
def match_litho(value):
    for key, group_name in litho_mapping.items():
        if key.lower() in value.lower():
            return group_name
    return "Other"

garonne_litho_fr_en = garonne_litho_fr.copy()
# Step 3: Apply to a new column
garonne_litho_fr_en["litho_group"] = garonne_litho_fr_en["libelleeh"].apply(match_litho)

To optimize the process it is important to dissolve the polygon geometries before intersecting the areas. Here we dissove it by the atribute field corresponding to the unique-id for each lithological class. 

In [20]:
attribute_field = 'litho_group'
litho_ff_dissolved = garonne_litho_fr_en.dissolve(by=attribute_field)

## Now we create a new feature with the lithology class:
litho_ff_dissolved["class"] = litho_ff_dissolved.index
litho_ff_dissolved

,geometry,basin_id,gauge_id,country,area_offic,area_estre,area_flag,area_rel,start_date,end_date,...,incluseh,etateh,natureeh,milieueh,themeeh,origineeh,layer_2,path_2,litho_name,class
litho_group,,,,,,,,,,,,,,,,,,,,,
Alluvium,"MULTIPOLYGON Z (((1.60659 42.86954 0.00000, 1....",FR001604,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,...,946,2,3,1,1,4,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,Alluvium
Carbonate rocks,"POLYGON Z ((2.10791 42.87875 0.00000, 2.10708 ...",FR001604,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,...,402,3,3,5,4,1,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,Carbonate rocks
Granitoids,"MULTIPOLYGON Z (((1.41376 42.64698 0.00000, 1....",FR001604,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,...,404,2,4,2,4,4,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,Granitoids
Impermeable,"MULTIPOLYGON Z (((0.37898 43.03226 0.00000, 0....",FR001604,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,...,400,3,4,4,4,4,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,Impermeable
Inland water,"POLYGON Z ((0.87458 42.62625 0.00000, 0.87458 ...",FR001604,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,...,None,None,None,None,None,None,8,C:\Users\nascimth\Downloads\8.shp,None,Inland water
Limestones,"MULTIPOLYGON Z (((0.81136 42.68557 0.00000, 0....",FR001604,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,...,699,2,3,3,4,1,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,Limestones
"Limestones, sandstones and gravels","MULTIPOLYGON Z (((1.85375 43.39243 0.00000, 1....",FR001604,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,...,334,3,3,4,2,1,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,"Limestones, sandstones and gravels"
"Limestones, sandstones and marls","MULTIPOLYGON Z (((2.08791 42.92030 0.00000, 2....",FR001604,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,...,681,3,4,4,4,1,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,"Limestones, sandstones and marls"
Magmatic rocks,"POLYGON Z ((1.00291 42.71125 0.00000, 1.00291 ...",FR001604,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,...,None,None,None,None,None,None,9,C:\Users\nascimth\Downloads\9.shp,None,Magmatic rocks


## Reproject to projected coordinates system

In [21]:
# Here you can check the crs of the datasets:
print("CRS of catchment_boundaries:", catchment_boundaries_clip.crs)
print("CRS of GLiM:", litho_ff_dissolved.crs)

CRS of catchment_boundaries: epsg:4326
CRS of GLiM: epsg:4326


In [22]:
# Define the target CRS to ETRS89 LAEA (3035)
target_crs = 'EPSG:3035'  

# Reproject the GeoDataFrame to the target CRS
catchment_boundaries_reprojected = catchment_boundaries_clip.to_crs(target_crs)
litho_ff_dissolved_reprojected = litho_ff_dissolved.to_crs(target_crs)

In [23]:
# Here you can check the crs of the datasets:
print("CRS of catchment_boundaries:", catchment_boundaries_reprojected.crs)
print("CRS of GLiM:", litho_ff_dissolved_reprojected.crs)

CRS of catchment_boundaries: EPSG:3035
CRS of GLiM: EPSG:3035


# Intersection areas

In [24]:
subset_catchment=catchment_boundaries_reprojected.copy()
subset_catchment["basin_id"] = subset_catchment.index
subset_catchment

,gauge_id,country,area_offic,area_estre,area_flag,area_rel,start_date,end_date,gauge_flag,upstream,group,geometry,basin_id
basin_id,,,,,,,,,,,,,
ES000333,9019,ES,440.0,445.690,0,-1.293182,1965-10-01,2019-09-30,E,3,541,"POLYGON Z ((3558169.009 2226716.483 0.000, 355...",ES000333
ES000453,9143,ES,137.0,144.472,0,-5.454015,1950-10-01,2018-09-30,E,1,541,"POLYGON Z ((3573647.634 2227227.141 0.000, 357...",ES000453
ES000507,9200,ES,50.0,145.186,999,-190.372000,1980-10-01,1991-10-27,B,2,541,"POLYGON Z ((3573647.634 2227227.141 0.000, 357...",ES000507
FR004084,O001531001,FR,38.0,31.038,1,18.321053,1961-01-01,2023-09-30,C,1,541,"POLYGON Z ((3567564.175 2244914.481 0.000, 356...",FR004084
FR004085,O004401001,FR,300.0,302.892,0,-0.964000,1920-01-01,1931-12-31,A,2,541,"POLYGON Z ((3549535.363 2243442.422 0.000, 354...",FR004085
...,...,...,...,...,...,...,...,...,...,...,...,...,...
FR001600,O243402001,FR,NaN,43.207,999,NaN,2014-01-01,2023-09-30,C,2,541,"POLYGON Z ((3561584.079 2288773.650 0.000, 356...",FR001600
FR001601,O243402101,FR,0.0,1.996,999,-inf,1970-10-01,1993-09-30,A,1,541,"POLYGON Z ((3547353.802 2278184.060 0.000, 354...",FR001601
FR001602,O246293201,FR,424.0,423.228,0,0.182075,1965-10-01,2006-02-21,A,6,541,"POLYGON Z ((3563216.203 2282328.895 0.000, 356...",FR001602


In [25]:
# Record the start time
start_time = time.time()

lithology_overlap = gpd.overlay(df1=subset_catchment, df2=litho_ff_dissolved_reprojected, how='intersection')

# Record the end time
end_time = time.time()

# Print the elapsed time in seconds when done:
print("Elapsed time: {:.1f} seconds".format(end_time - start_time))

Elapsed time: 5.3 seconds


In [26]:
# Calculate the areas of the overlapping polygons and add them as a new column
lithology_overlap['area_sqm'] = lithology_overlap['geometry'].area/1000000
lithology_overlap

,gauge_id_1,country_1,area_offic_1,area_estre_1,area_flag_1,area_rel_1,start_date_1,end_date_1,gauge_flag_1,upstream_1,...,natureeh,milieueh,themeeh,origineeh,layer_2,path_2,litho_name,class,geometry,area_sqm
0,9019,ES,440.0,445.690,0,-1.293182,1965-10-01,2019-09-30,E,3,...,None,None,None,None,8,C:\Users\nascimth\Downloads\8.shp,None,Inland water,"POLYGON Z ((3572117.650 2214386.439 0.000, 357...",0.970290
1,9019,ES,440.0,445.690,0,-1.293182,1965-10-01,2019-09-30,E,3,...,3,3,4,1,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,Limestones,"POLYGON Z ((3567903.309 2221504.293 0.000, 356...",26.738209
2,9019,ES,440.0,445.690,0,-1.293182,1965-10-01,2019-09-30,E,3,...,None,None,None,None,9,C:\Users\nascimth\Downloads\9.shp,None,Magmatic rocks,"POLYGON Z ((3583662.597 2222663.741 0.000, 358...",0.953672
3,9019,ES,440.0,445.690,0,-1.293182,1965-10-01,2019-09-30,E,3,...,4,2,4,4,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,Massif,MULTIPOLYGON Z (((3578438.554 2231855.836 0.00...,0.411684
4,9019,ES,440.0,445.690,0,-1.293182,1965-10-01,2019-09-30,E,3,...,None,None,None,None,2,C:\Users\nascimth\Downloads\2.shp,None,Plutonic rocks,MULTIPOLYGON Z (((3582253.904 2218088.368 0.00...,158.244339
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
859,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,C,187,...,4,1,2,4,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,Sands and clays,MULTIPOLYGON Z (((3606018.347 2268524.209 0.00...,0.343879
860,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,C,187,...,4,4,4,1,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,"Schists, gneisses, granites",MULTIPOLYGON Z (((3664239.128 2207472.389 0.00...,27.876651
861,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,C,187,...,4,1,2,4,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,Sedimentary rocks,MULTIPOLYGON Z (((3675715.383 2247028.602 0.00...,0.748232
862,O262002002,FR,13730.0,13752.051,0,-0.160605,1972-08-01,2023-09-30,C,187,...,4,4,4,4,litho_fr_n2_o1_3035,C:/Users/nascimth/Documents/data/gis/litho_fr_...,None,Sediments mesosoics,MULTIPOLYGON Z (((3639005.959 2238484.131 0.00...,267.336581


In [27]:
lithology_overlap.columns

Index(['gauge_id_1', 'country_1', 'area_offic_1', 'area_estre_1',
       'area_flag_1', 'area_rel_1', 'start_date_1', 'end_date_1',
       'gauge_flag_1', 'upstream_1', 'group_1', 'basin_id_1', 'basin_id_2',
       'gauge_id_2', 'country_2', 'area_offic_2', 'area_estre_2',
       'area_flag_2', 'area_rel_2', 'start_date_2', 'end_date_2',
       'gauge_flag_2', 'upstream_2', 'group_2', 'layer', 'path', 'LEVEL1',
       'LEVEL2', 'LEVEL3', 'LEVEL4', 'LEVEL5', 'Shape_Leng', 'Shape_Area',
       'codeeh', 'ordreabseh', 'ordrereleh', 'niveaueh', 'libelleeh',
       'incluseh', 'etateh', 'natureeh', 'milieueh', 'themeeh', 'origineeh',
       'layer_2', 'path_2', 'litho_name', 'class', 'geometry', 'area_sqm'],
      dtype='object')

# Pivot table

In [28]:
# Finally we can creatre a pivot-table with the percentage of each lithological class per catchment:

lithology_areas = pd.pivot_table(
    lithology_overlap,
    values='area_sqm',     # Replace with the actual column name for the area
    index='basin_id_1',      # Rows are based on 'basin_id'
    columns='class',       # Columns are based on 'class' (the class)
    aggfunc='sum',         # Sum the areas for each combination
    fill_value=0           # Replace NaN with 0
)

# Here we can sum to compute the total area of each catchment: 
lithology_areas.loc[:, "totalarea"] = lithology_areas.sum(axis = 1)
lithology_areas.index.name = "basin_id"
lithology_areas

class,Alluvium,Carbonate rocks,Granitoids,Impermeable,Inland water,Limestones,"Limestones, sandstones and gravels","Limestones, sandstones and marls",Magmatic rocks,Massif,Molasses and clays,Plutonic rocks,"Sand, gravel and pebbles",Sands and clays,"Schists, gneisses, granites",Sedimentary rocks,Sediments mesosoics,Shales,totalarea
basin_id,,,,,,,,,,,,,,,,,,,
ES000333,0.000000,0.0,0.0,0.000000,0.97029,26.738209,0.000000,0.0,0.953672,0.411684,0.000000,158.244339,0.000000,0.0,0.0,0.00000,0.0,258.325491,445.643686
ES000453,0.000000,0.0,0.0,0.000000,0.00000,2.918189,0.000000,0.0,0.953672,0.036509,0.000000,69.621166,0.000000,0.0,0.0,0.00000,0.0,70.602811,144.132349
ES000507,0.000000,0.0,0.0,0.000000,0.00000,2.918189,0.000000,0.0,0.953672,0.036509,0.000000,69.621166,0.000000,0.0,0.0,0.00000,0.0,71.317514,144.847052
FR001495,0.000000,0.0,0.0,0.000000,0.97029,26.738209,0.000000,0.0,0.953672,3.473942,0.000000,170.627130,0.000000,0.0,0.0,0.00000,0.0,354.653011,557.416254
FR001496,0.000000,0.0,0.0,0.000000,0.97029,26.738209,0.000000,0.0,0.953672,0.913608,0.000000,170.627130,0.000000,0.0,0.0,0.00000,0.0,350.806543,551.009452
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FR004159,0.000000,0.0,0.0,0.000000,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,30.770798,0.000000,0.000000,0.0,0.0,0.17945,0.0,0.000000,30.950248
FR004160,0.000000,0.0,0.0,0.000000,0.00000,0.000000,0.000392,0.0,0.000000,0.000000,524.267622,0.000000,1.580755,0.0,0.0,0.00000,0.0,0.000000,525.848770
FR004161,0.000000,0.0,0.0,98.123483,0.00000,84.674257,0.000000,0.0,0.000000,0.000000,242.309804,0.000000,16.158228,0.0,0.0,0.00000,0.0,0.000000,441.265771


## Catchment covered by shapefile
* Here we compute the total catchment area covered by the lithology shapefile:


In [29]:
#catchment_boundaries_reprojected.set_index('basin_id', inplace = True)

In [30]:
lithology_areas['area_calc'] = catchment_boundaries_reprojected.area / 1000000
lithology_areas

class,Alluvium,Carbonate rocks,Granitoids,Impermeable,Inland water,Limestones,"Limestones, sandstones and gravels","Limestones, sandstones and marls",Magmatic rocks,Massif,Molasses and clays,Plutonic rocks,"Sand, gravel and pebbles",Sands and clays,"Schists, gneisses, granites",Sedimentary rocks,Sediments mesosoics,Shales,totalarea,area_calc
basin_id,,,,,,,,,,,,,,,,,,,,
ES000333,0.000000,0.0,0.0,0.000000,0.97029,26.738209,0.000000,0.0,0.953672,0.411684,0.000000,158.244339,0.000000,0.0,0.0,0.00000,0.0,258.325491,445.643686,445.689746
ES000453,0.000000,0.0,0.0,0.000000,0.00000,2.918189,0.000000,0.0,0.953672,0.036509,0.000000,69.621166,0.000000,0.0,0.0,0.00000,0.0,70.602811,144.132349,144.471757
ES000507,0.000000,0.0,0.0,0.000000,0.00000,2.918189,0.000000,0.0,0.953672,0.036509,0.000000,69.621166,0.000000,0.0,0.0,0.00000,0.0,71.317514,144.847052,145.186460
FR001495,0.000000,0.0,0.0,0.000000,0.97029,26.738209,0.000000,0.0,0.953672,3.473942,0.000000,170.627130,0.000000,0.0,0.0,0.00000,0.0,354.653011,557.416254,554.400056
FR001496,0.000000,0.0,0.0,0.000000,0.97029,26.738209,0.000000,0.0,0.953672,0.913608,0.000000,170.627130,0.000000,0.0,0.0,0.00000,0.0,350.806543,551.009452,550.553588
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FR004159,0.000000,0.0,0.0,0.000000,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,30.770798,0.000000,0.000000,0.0,0.0,0.17945,0.0,0.000000,30.950248,30.950250
FR004160,0.000000,0.0,0.0,0.000000,0.00000,0.000000,0.000392,0.0,0.000000,0.000000,524.267622,0.000000,1.580755,0.0,0.0,0.00000,0.0,0.000000,525.848770,525.849148
FR004161,0.000000,0.0,0.0,98.123483,0.00000,84.674257,0.000000,0.0,0.000000,0.000000,242.309804,0.000000,16.158228,0.0,0.0,0.00000,0.0,0.000000,441.265771,441.265773


In [31]:
lithology_areas['tot_area'] = lithology_areas.totalarea / lithology_areas.area_calc
lithology_areas

class,Alluvium,Carbonate rocks,Granitoids,Impermeable,Inland water,Limestones,"Limestones, sandstones and gravels","Limestones, sandstones and marls",Magmatic rocks,Massif,...,Plutonic rocks,"Sand, gravel and pebbles",Sands and clays,"Schists, gneisses, granites",Sedimentary rocks,Sediments mesosoics,Shales,totalarea,area_calc,tot_area
basin_id,,,,,,,,,,,,,,,,,,,,,
ES000333,0.000000,0.0,0.0,0.000000,0.97029,26.738209,0.000000,0.0,0.953672,0.411684,...,158.244339,0.000000,0.0,0.0,0.00000,0.0,258.325491,445.643686,445.689746,0.999897
ES000453,0.000000,0.0,0.0,0.000000,0.00000,2.918189,0.000000,0.0,0.953672,0.036509,...,69.621166,0.000000,0.0,0.0,0.00000,0.0,70.602811,144.132349,144.471757,0.997651
ES000507,0.000000,0.0,0.0,0.000000,0.00000,2.918189,0.000000,0.0,0.953672,0.036509,...,69.621166,0.000000,0.0,0.0,0.00000,0.0,71.317514,144.847052,145.186460,0.997662
FR001495,0.000000,0.0,0.0,0.000000,0.97029,26.738209,0.000000,0.0,0.953672,3.473942,...,170.627130,0.000000,0.0,0.0,0.00000,0.0,354.653011,557.416254,554.400056,1.005440
FR001496,0.000000,0.0,0.0,0.000000,0.97029,26.738209,0.000000,0.0,0.953672,0.913608,...,170.627130,0.000000,0.0,0.0,0.00000,0.0,350.806543,551.009452,550.553588,1.000828
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FR004159,0.000000,0.0,0.0,0.000000,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.17945,0.0,0.000000,30.950248,30.950250,1.000000
FR004160,0.000000,0.0,0.0,0.000000,0.00000,0.000000,0.000392,0.0,0.000000,0.000000,...,0.000000,1.580755,0.0,0.0,0.00000,0.0,0.000000,525.848770,525.849148,0.999999
FR004161,0.000000,0.0,0.0,98.123483,0.00000,84.674257,0.000000,0.0,0.000000,0.000000,...,0.000000,16.158228,0.0,0.0,0.00000,0.0,0.000000,441.265771,441.265773,1.000000


# Data organization

In [34]:
# Here we compute the lithology percentages from each class:
lithology_df = (lithology_areas.iloc[:, 0:-2].div(lithology_areas['totalarea'], axis=0))*100
#lithology_df = lithology_df.iloc[:, 0:-2]

lithology_df

class,Alluvium,Carbonate rocks,Granitoids,Impermeable,Inland water,Limestones,"Limestones, sandstones and gravels","Limestones, sandstones and marls",Magmatic rocks,Massif,Molasses and clays,Plutonic rocks,"Sand, gravel and pebbles",Sands and clays,"Schists, gneisses, granites",Sedimentary rocks,Sediments mesosoics,Shales,totalarea
basin_id,,,,,,,,,,,,,,,,,,,
ES000333,0.000000,0.0,0.0,0.000000,0.217728,5.999908,0.000000,0.0,0.213999,0.092380,0.000000,35.509162,0.000000,0.0,0.0,0.000000,0.0,57.966824,100.0
ES000453,0.000000,0.0,0.0,0.000000,0.000000,2.024659,0.000000,0.0,0.661664,0.025330,0.000000,48.303637,0.000000,0.0,0.0,0.000000,0.0,48.984709,100.0
ES000507,0.000000,0.0,0.0,0.000000,0.000000,2.014669,0.000000,0.0,0.658400,0.025205,0.000000,48.065298,0.000000,0.0,0.0,0.000000,0.0,49.236428,100.0
FR001495,0.000000,0.0,0.0,0.000000,0.174069,4.796812,0.000000,0.0,0.171088,0.623222,0.000000,30.610361,0.000000,0.0,0.0,0.000000,0.0,63.624447,100.0
FR001496,0.000000,0.0,0.0,0.000000,0.176093,4.852586,0.000000,0.0,0.173077,0.165806,0.000000,30.966280,0.000000,0.0,0.0,0.000000,0.0,63.666157,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FR004159,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,99.420198,0.000000,0.000000,0.0,0.0,0.579802,0.0,0.000000,100.0
FR004160,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000075,0.0,0.000000,0.000000,99.699315,0.000000,0.300610,0.0,0.0,0.000000,0.0,0.000000,100.0
FR004161,0.000000,0.0,0.0,22.236822,0.000000,19.188947,0.000000,0.0,0.000000,0.000000,54.912440,0.000000,3.661791,0.0,0.0,0.000000,0.0,0.000000,100.0


In [35]:
# Create a new column with the name of the column with the majority class
lithology_df['lit_dom'] = lithology_df.iloc[:, 0:-1].apply(lambda row: row.idxmax(), axis=1)

# Add "th_new_" as a prefix to all column names
lithology_df = lithology_df.add_prefix('lit_fra_')
lithology_df = lithology_df.rename(columns={'lit_fra_lit_dom': 'lit_dom'})

# Catchment ara covered by the lithology shapef
lithology_df['tot_area'] = (lithology_areas.tot_area)*100

lithology_df

class,lit_fra_Alluvium,lit_fra_Carbonate rocks,lit_fra_Granitoids,lit_fra_Impermeable,lit_fra_Inland water,lit_fra_Limestones,"lit_fra_Limestones, sandstones and gravels","lit_fra_Limestones, sandstones and marls",lit_fra_Magmatic rocks,lit_fra_Massif,...,lit_fra_Plutonic rocks,"lit_fra_Sand, gravel and pebbles",lit_fra_Sands and clays,"lit_fra_Schists, gneisses, granites",lit_fra_Sedimentary rocks,lit_fra_Sediments mesosoics,lit_fra_Shales,lit_fra_totalarea,lit_dom,tot_area
basin_id,,,,,,,,,,,,,,,,,,,,,
ES000333,0.000000,0.0,0.0,0.000000,0.217728,5.999908,0.000000,0.0,0.213999,0.092380,...,35.509162,0.000000,0.0,0.0,0.000000,0.0,57.966824,100.0,Shales,99.989665
ES000453,0.000000,0.0,0.0,0.000000,0.000000,2.024659,0.000000,0.0,0.661664,0.025330,...,48.303637,0.000000,0.0,0.0,0.000000,0.0,48.984709,100.0,Shales,99.765070
ES000507,0.000000,0.0,0.0,0.000000,0.000000,2.014669,0.000000,0.0,0.658400,0.025205,...,48.065298,0.000000,0.0,0.0,0.000000,0.0,49.236428,100.0,Shales,99.766226
FR001495,0.000000,0.0,0.0,0.000000,0.174069,4.796812,0.000000,0.0,0.171088,0.623222,...,30.610361,0.000000,0.0,0.0,0.000000,0.0,63.624447,100.0,Shales,100.544047
FR001496,0.000000,0.0,0.0,0.000000,0.176093,4.852586,0.000000,0.0,0.173077,0.165806,...,30.966280,0.000000,0.0,0.0,0.000000,0.0,63.666157,100.0,Shales,100.082801
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FR004159,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.579802,0.0,0.000000,100.0,Molasses and clays,99.999994
FR004160,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000075,0.0,0.000000,0.000000,...,0.000000,0.300610,0.0,0.0,0.000000,0.0,0.000000,100.0,Molasses and clays,99.999928
FR004161,0.000000,0.0,0.0,22.236822,0.000000,19.188947,0.000000,0.0,0.000000,0.000000,...,0.000000,3.661791,0.0,0.0,0.000000,0.0,0.000000,100.0,Molasses and clays,100.000000


In [36]:
# Here we sort the index:
lithology_df = lithology_df.sort_index(axis=0)
lithology_df

class,lit_fra_Alluvium,lit_fra_Carbonate rocks,lit_fra_Granitoids,lit_fra_Impermeable,lit_fra_Inland water,lit_fra_Limestones,"lit_fra_Limestones, sandstones and gravels","lit_fra_Limestones, sandstones and marls",lit_fra_Magmatic rocks,lit_fra_Massif,...,lit_fra_Plutonic rocks,"lit_fra_Sand, gravel and pebbles",lit_fra_Sands and clays,"lit_fra_Schists, gneisses, granites",lit_fra_Sedimentary rocks,lit_fra_Sediments mesosoics,lit_fra_Shales,lit_fra_totalarea,lit_dom,tot_area
basin_id,,,,,,,,,,,,,,,,,,,,,
ES000333,0.000000,0.0,0.0,0.000000,0.217728,5.999908,0.000000,0.0,0.213999,0.092380,...,35.509162,0.000000,0.0,0.0,0.000000,0.0,57.966824,100.0,Shales,99.989665
ES000453,0.000000,0.0,0.0,0.000000,0.000000,2.024659,0.000000,0.0,0.661664,0.025330,...,48.303637,0.000000,0.0,0.0,0.000000,0.0,48.984709,100.0,Shales,99.765070
ES000507,0.000000,0.0,0.0,0.000000,0.000000,2.014669,0.000000,0.0,0.658400,0.025205,...,48.065298,0.000000,0.0,0.0,0.000000,0.0,49.236428,100.0,Shales,99.766226
FR001495,0.000000,0.0,0.0,0.000000,0.174069,4.796812,0.000000,0.0,0.171088,0.623222,...,30.610361,0.000000,0.0,0.0,0.000000,0.0,63.624447,100.0,Shales,100.544047
FR001496,0.000000,0.0,0.0,0.000000,0.176093,4.852586,0.000000,0.0,0.173077,0.165806,...,30.966280,0.000000,0.0,0.0,0.000000,0.0,63.666157,100.0,Shales,100.082801
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FR004159,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.579802,0.0,0.000000,100.0,Molasses and clays,99.999994
FR004160,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000075,0.0,0.000000,0.000000,...,0.000000,0.300610,0.0,0.0,0.000000,0.0,0.000000,100.0,Molasses and clays,99.999928
FR004161,0.000000,0.0,0.0,22.236822,0.000000,19.188947,0.000000,0.0,0.000000,0.000000,...,0.000000,3.661791,0.0,0.0,0.000000,0.0,0.000000,100.0,Molasses and clays,100.000000


In [37]:
# Round the data to 3 decimals:
lithology_df.iloc[:, 0:-3] = lithology_df.iloc[:, 0:-3].round(3)
lithology_df.iloc[:, -2:] = lithology_df.iloc[:, -2:].round(3)
lithology_df.drop("lit_fra_totalarea", axis = 1, inplace = True)
lithology_df

class,lit_fra_Alluvium,lit_fra_Carbonate rocks,lit_fra_Granitoids,lit_fra_Impermeable,lit_fra_Inland water,lit_fra_Limestones,"lit_fra_Limestones, sandstones and gravels","lit_fra_Limestones, sandstones and marls",lit_fra_Magmatic rocks,lit_fra_Massif,lit_fra_Molasses and clays,lit_fra_Plutonic rocks,"lit_fra_Sand, gravel and pebbles",lit_fra_Sands and clays,"lit_fra_Schists, gneisses, granites",lit_fra_Sedimentary rocks,lit_fra_Sediments mesosoics,lit_fra_Shales,lit_dom,tot_area
basin_id,,,,,,,,,,,,,,,,,,,,
ES000333,0.000,0.0,0.0,0.000,0.218,6.000,0.0,0.0,0.214,0.092,0.000,35.509,0.000,0.0,0.0,0.00,0.0,57.967,Shales,99.990
ES000453,0.000,0.0,0.0,0.000,0.000,2.025,0.0,0.0,0.662,0.025,0.000,48.304,0.000,0.0,0.0,0.00,0.0,48.985,Shales,99.765
ES000507,0.000,0.0,0.0,0.000,0.000,2.015,0.0,0.0,0.658,0.025,0.000,48.065,0.000,0.0,0.0,0.00,0.0,49.236,Shales,99.766
FR001495,0.000,0.0,0.0,0.000,0.174,4.797,0.0,0.0,0.171,0.623,0.000,30.610,0.000,0.0,0.0,0.00,0.0,63.624,Shales,100.544
FR001496,0.000,0.0,0.0,0.000,0.176,4.853,0.0,0.0,0.173,0.166,0.000,30.966,0.000,0.0,0.0,0.00,0.0,63.666,Shales,100.083
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FR004159,0.000,0.0,0.0,0.000,0.000,0.000,0.0,0.0,0.000,0.000,99.420,0.000,0.000,0.0,0.0,0.58,0.0,0.000,Molasses and clays,100.000
FR004160,0.000,0.0,0.0,0.000,0.000,0.000,0.0,0.0,0.000,0.000,99.699,0.000,0.301,0.0,0.0,0.00,0.0,0.000,Molasses and clays,100.000
FR004161,0.000,0.0,0.0,22.237,0.000,19.189,0.0,0.0,0.000,0.000,54.912,0.000,3.662,0.0,0.0,0.00,0.0,0.000,Molasses and clays,100.000


# Data export

In [38]:
# Export the final dataset:
lithology_df.to_csv("../../results/staticattributes//garonne_estreams_geologyregional_attributes.csv")

# End